In [ ]:
#!pip install openai

In [8]:
import json
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI

In [9]:
OPENAI_MODEL = "gpt-4o-mini" #gpt-5-mini

In [10]:
SYSTEM_PROMPT = (
    "Sei un estrattore di informazioni. "
    "Devi rispondere SOLO in formato json valido (oggetto JSON), senza testo extra, "
    "senza markdown e senza code fences. "
    "La risposta deve essere un unico oggetto json."
)

USER_INSTRUCTIONS = """Estrai l'elenco strutturato dei centri/sportelli/case citati nel testo qui sotto.
Regole:
- 'tipo' ∈ {Centro Antiviolenza, Sportello collegato, Casa Rifugio, Altro}
- Compila comuni/indirizzi solo se esplicitamente presenti.
- Indica in 'ente_capofila' se dal testo emerge (es. “Comune di Bra”).
- In 'note' aggiungi contesto utile (es. “sportelli decentrati del CAV n.10/A del Cuneese”, “collegamento al 1522”, “nuovo centro”).
Testo:
"""

In [4]:
testo_protocollo = """PROTOCOLLO TERRITORIALE TRA COMUNE DI BRA, IL COMUNE DI ALBA, IL CONSORZIO
SOCIO ASSISTENZIALE ALBA LANGHE ROERO, L’A.S.L CN 2 E L;ASSOCIAZIONE MAI+SOLE,
PER LA REALIZZAZIONE DI UN NUOVO CENTRO ANTIVIOLENZA

TRA

- Il Comune di Bra (Ente capofila), in qualita di Ente Locale territoriale e soggetto gestore dei Servizi Socio
Assistenziali, con sede legale a Bra, piazza Caduti per la Liberta n. 14, C.F. n. 82000150043/ P.Iva n.
00493130041, rappresentato dal Sindaco Giovanni Fogliato nato a Bra il 24/09/1961 domiciliato, ai fini del
presente accordo, presso la sede legale dell’ Ente;

E

- Il Consorzio Socio Assistenziale Alba Langhe Roero, con sede legale ad Alba (CN), Via A. Diaz, n. 8, CF/
P. IVA: n. 02797980048, legalmente rappresentato dalla dott.ssa Loredana Defilippi n. ad Alba il
23/06/1960 , domiciliata ai fini del presente accordo presso la sede legale dell’ Ente;

E

- Il Comune di Alba, con sede legale ad Alba in Piazza Risorgimento n. 1 C.F/P.IVA 00184260040,
rappresentato dal sindaco pro tempore CARLO BO, nato Carmagnola (TO) il 15/08/1970, domicliato, ai
fine del presente accordo presso la sede legale dell'Ente;

E

LASL CN2 , con sede legale ad Alba (CN) Via Vida n. 10, CF/PIVA n. 02419170044, rappresentata dal
legale rappresentante Massimo VEGLIO nato Torino il 18/07/1959, domiciliato ai fini del presente accordo
presso la sede legale dell’ Ente;

E

- L’Associazione MAI+SOLE, con — sede legale in Savigliano (CN), P. IVA/C.F: n. 95019460047
rappresentata dal legale rappresentante: FIORITO Adonella, nata a Villafalletto (CN) il 20/09/1956,
domiciliata ai fini del presente accordo a Savigliano in Via Teatro n. 2;

RICHIAMATI

e la Legge 27 giugno 2013 n.77 “Ratifica ed esecuzione della Convenzione del Consiglio d’Europa sulla
prevenzione e la lotta contro la violenza nei confronti delle donne e la violenza domestica, fatta ad
Istanbul I’11 maggio 2011”;

e la Legge 15 ottobre 2013, n. 119, “Conversione in legge, con modificazioni, del decreto-legge 14 agosto
2013, n. 93, recante disposizioni urgenti in materia di sicurezza e per il contrasto della violenza di
genere, nonché in tema di protezione civile e di commissariamento delle province”, che individua, tra gli
obiettivi di cui all’art. 5, comma 2, “d) potenziare le forme di assistenza e di sostegno alle donne vittime
di violenza e ai loro figli attraverso modalita omogenee di rafforzamento della rete dei servizi
territoriali, dei centri antiviolenza e dei servizi di assistenza alle donne vittime di violenza”;

e ’Intesa CU n. 146 del 27 novembre 2014, tra il Governo e le Regioni, le Province autonome di Trento e
di Bolzano e le Autonomie locali, relativa ai requisiti minimi dei Centri antiviolenza e delle Case
Rifugio;

e la Legge regionale 18 marzo 2009, n. 8, “Integrazione delle politiche di pari opportunita di genere nella
Regione Piemonte e disposizioni per l'istituzione dei bilanci di genere”, che all’articolo 2, comma h)
recita: “promuovere e sostenere azioni volte a prevenire la violenza fondata sul genere e la tratta delle
donne, anche attivando piani e programmi per la tutela delle vittime’’;

1

Comune di Bra
Il Sindaco

Comune di Alba
II Sindaco

Consorzio Socio Assistenziale Alba Langhe Roero
Il Presidente Dr.ssa Loredana Defilippi

ASL 
Il Direttore Generale 

Associazione MAI+SOLE
Il legale Rappresentante 
"""

In [11]:
import os
import json

cartella_txt = "script/txtoren"

# 1️⃣ Carica tutti i file .txt in un array
testi = []
file_txt = [f for f in os.listdir(cartella_txt) if f.endswith(".txt")]

for nome_file in file_txt:
    percorso = os.path.join(cartella_txt, nome_file)
    with open(percorso, "r", encoding="utf-8") as f:
        testo = f.read()
        testi.append({"nome_file": nome_file, "contenuto": testo})

print(f"📂 Caricati {len(testi)} file di testo.")

📂 Caricati 23 file di testo.


In [12]:
# Chiave API
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [13]:
risultati = []

for item in testi:
    try:
        resp = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_PROMPT,
            input=USER_INSTRUCTIONS + item["contenuto"] + "\n---"
        )
        data = json.loads(resp.output_text)
        risultati.append({
            "file": item["nome_file"],
            "risultato": data
        })
    except Exception as e:
        print(f"❌ Errore con {item['nome_file']}: {e}")
        risultati.append({
            "file": item["nome_file"],
            "risultato": None
        })

# 3️⃣ Stampa i risultati (primi 2 come esempio)
print(json.dumps(risultati[:2], indent=4, ensure_ascii=False))

[
    {
        "file": "01_2406_rta_01_240904110730_4166---protocollocontuttelefirme(1).txt",
        "risultato": {
            "centri": [
                {
                    "nome": "Centro Antiviolenza",
                    "tipo": "Centro Antiviolenza",
                    "comune": "Bra",
                    "indirizzo": "Piazza Caduti per la Libertà n. 14",
                    "ente_capofila": "Comune di Bra",
                    "note": "Nuovo centro antiviolenza in via di attivazione"
                },
                {
                    "nome": "Centro Antiviolenza",
                    "tipo": "Centro Antiviolenza",
                    "comune": "Alba",
                    "indirizzo": "Piazza Risorgimento n. 1",
                    "ente_capofila": "Comune di Bra",
                    "note": "Contributo nella gestione del CAV"
                },
                {
                    "nome": "CAV n. 10/A",
                    "tipo": "Altro",
                    "comu

In [14]:
import json
import os

# 📂 Cartella di output
output_dir = "script/output/json"
os.makedirs(output_dir, exist_ok=True)

# 📄 Salva tutti i risultati in un unico file
output_file = os.path.join(output_dir, "risultati.json")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(risultati, f, indent=4, ensure_ascii=False)

print(f"✅ Risultati salvati in: {output_file}")


✅ Risultati salvati in: script/output/json\risultati.json


In [7]:
resp = client.responses.create(
    model=OPENAI_MODEL,
    instructions=SYSTEM_PROMPT,
    input=USER_INSTRUCTIONS + testo_protocollo + "\n---"
)
data = json.loads(resp.output_text)
print(json.dumps(data, indent=4, ensure_ascii=False))

{
    "centri": [
        {
            "tipo": "Centro Antiviolenza",
            "comune": "Bra",
            "indirizzo": "Piazza Caduti per la Liberta n. 14",
            "ente_capofila": "Comune di Bra",
            "note": "nuovo centro"
        },
        {
            "tipo": "Centro Antiviolenza",
            "comune": "Alba",
            "indirizzo": "Piazza Risorgimento n. 1",
            "ente_capofila": "Comune di Alba",
            "note": "nuovo centro"
        },
        {
            "tipo": "Altro",
            "comune": "Alba",
            "indirizzo": "Via A. Diaz n. 8",
            "ente_capofila": "Consorzio Socio Assistenziale Alba Langhe Roero",
            "note": "gestore dei Servizi Socio Assistenziali"
        },
        {
            "tipo": "Altro",
            "comune": "Alba",
            "indirizzo": "Via Vida n. 10",
            "ente_capofila": "ASL CN2",
            "note": "gestione servizi sanitari"
        },
        {
            "tipo": "Altro",